# S10 · One decision tree, and how it overfits

A lending app has to decide, in seconds, whether a new applicant will repay a
loan. We build the simplest model that can make that decision: a **decision
tree**, a flowchart of yes/no questions. We draw it, read it, and then watch what
goes wrong when we let it grow too deep.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain-English note above each one.
- Already confident with code or with trees? Look for the cells marked
  **Stretch (optional)**.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

Run the next cell to load the libraries and the dataset. It works in **Google
Colab** (where the file is at `/content/`) and **on your own machine** (where it
sits in the session's `data/` folder).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def load(name):
    """Load one of this session's teaching datasets."""
    local = Path("../data") / f"{name}.csv"
    if local.exists():
        return pd.read_csv(local)
    return pd.read_csv(f"/content/{name}.csv")


print("Setup complete.")

## Step 1 — meet the loan applicants

The data is a small, made-up set of past loan applicants. For each one we know
two numbers: their monthly **income** and their **credit score**. What we want to
predict is whether they **repay** the loan (`1`) or **default** (`0`).

In [ ]:
data = load("tree_loan_data")
data.head()

## Step 2 — always look at the data first

Before any modelling, plot the applicants and just look. Each dot is one person:
income across the bottom, credit score up the side. Blue dots repaid, red dots
defaulted. The tree's whole job is to separate the colours by drawing boundaries.

In [ ]:
repaid = data[data["repaid"] == 1]
defaulted = data[data["repaid"] == 0]

plt.figure(figsize=(7, 5))
plt.scatter(repaid["income"], repaid["credit_score"],
            color="#2E75B6", label="repaid")
plt.scatter(defaulted["income"], defaulted["credit_score"],
            color="#C0392B", label="defaulted")
plt.xlabel("monthly income (1000s of rupees)")
plt.ylabel("credit score")
plt.title("Our loan applicants")
plt.legend()
plt.show()

## Step 3 — put the two inputs into one table

`scikit-learn`, the standard machine-learning library, wants the inputs as a
table called `X`: one row per applicant, one column per input. The answers (repaid
or not) go in a matching list called `y`.

In [ ]:
X = data[["income", "credit_score"]]
y = data["repaid"]

feature_names = ["income", "credit_score"]

print("X shape:", X.shape, "  (rows = people, columns = inputs)")
print("y shape:", y.shape, "  (one answer per person)")

## Step 4 — split into a part to learn from and a part to test on

We train the tree on one part of the data and keep another part hidden. Then we
check the tree on people it has never seen. This train/test habit is the single
most important discipline in machine learning: it is the only honest way to know
whether a model actually works or has just memorised.

In [ ]:
from sklearn.model_selection import train_test_split

# 70% for training, 30% kept aside for an honest test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

print("training applicants:", X_train.shape[0])
print("test applicants    :", X_test.shape[0])

## Step 5 — grow a shallow decision tree

A decision tree is a flowchart of yes/no questions. We grow a shallow one for now
(`max_depth=2` means it may ask at most two questions before deciding) so it is
easy to read. The tree picks each question to make the two groups it creates as
**tidy** as possible: ideally one group is nearly all 'repaid' and the other
nearly all 'defaulted'.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# max_depth=2: the tree may ask at most 2 questions before it decides.
small_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
small_tree.fit(X_train, y_train)

print("Tree trained.")

## Step 6 — draw the tree as a flowchart

Now the payoff: we can actually see the flowchart the tree learned. Read it top to
bottom. Start at the top box, answer its yes/no question, follow the branch, and
keep going until you reach a bottom box (called a **leaf**) that gives a verdict.

Each box also shows a number called `gini`. That is the impurity: how mixed the
group is. `gini = 0` means a perfectly tidy group, all one class. This readability
is a big reason trees are trusted for lending decisions: you can show anyone
exactly why an applicant was refused.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(11, 6))
plot_tree(small_tree,
          feature_names=feature_names,
          class_names=["default", "repaid"],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title("Our decision tree (depth 2)")
plt.show()

## Step 7 — score the shallow tree on the hidden test data

Accuracy is the fraction of applicants the tree labels correctly. We look at two
numbers: the score on the training data (the people it learned from) and the score
on the hidden test data (people it has never seen). The test score is the honest
one.

In [ ]:
train_accuracy = small_tree.score(X_train, y_train)
test_accuracy = small_tree.score(X_test, y_test)

print("shallow tree (depth 2)")
print("  training accuracy:", round(train_accuracy, 3))
print("  test accuracy    :", round(test_accuracy, 3))

## Step 8 — now let the tree grow as deep as it likes

Here is the important experiment. We remove the depth limit. The tree will keep
asking questions until every leaf is perfectly tidy. It will score a perfect
`1.0` on the training data. That sounds great, but it is a warning sign, not a
success: a tree that memorises every training applicant has stopped learning the
general pattern.

In [ ]:
# max_depth=None means "grow until every leaf is pure".
deep_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
deep_tree.fit(X_train, y_train)

deep_train_accuracy = deep_tree.score(X_train, y_train)
deep_test_accuracy = deep_tree.score(X_test, y_test)

print("deep tree (no depth limit)")
print("  training accuracy:", round(deep_train_accuracy, 3))
print("  test accuracy    :", round(deep_test_accuracy, 3))

## Step 9 — try every depth and find where the test score peaks

Let us sweep the depth from 1 to 12 and record both scores at each depth. We
expect the **training** score to keep climbing toward 1.0, while the **test**
score rises, peaks, and then sags as the tree begins memorising noise. That growing
gap between the two is called **overfitting**.

In [ ]:
depths_to_try = range(1, 13)

train_scores = []
test_scores = []

for depth in depths_to_try:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))

print("depth | train acc | test acc")
for depth, train_acc, test_acc in zip(depths_to_try, train_scores, test_scores):
    print("  ", depth, "  |  ", round(train_acc, 3), "  |  ", round(test_acc, 3))

## Step 10 — the picture that is the whole lesson

This one plot says it all. The training line keeps rising. The test line rises,
peaks, and then falls. The best tree is at the **peak of the test line**, not the
deepest one. Reining a tree in like this, so it stops before it memorises, is
called **pruning**.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(depths_to_try), train_scores,
         color="#2E75B6", marker="o", label="training accuracy")
plt.plot(list(depths_to_try), test_scores,
         color="#C0392B", marker="o", label="test accuracy")
plt.xlabel("max_depth (how many questions deep the tree may go)")
plt.ylabel("accuracy")
plt.title("Deeper is not better: the test score peaks, then overfits")
plt.legend()
plt.show()

### Stretch (optional) — how the tree measures a "tidy" group

Skip this if you are new to the maths; you already have the main idea. The `gini`
number in the tree boxes measures how mixed a group is. Here we compute both
common impurity measures, **Gini** and **entropy**, for a few example mixes, so
you can see they tell the same story: `0` for a pure group, biggest at a 50/50
mix.

`p` is the fraction of a group that repaid, `q = 1 - p` the fraction that
defaulted.

- Gini impurity: `1 - (p² + q²)`.
- Entropy (from information theory): `-p·log₂p - q·log₂q`.

In [ ]:
def impurity_scores(fraction_repaid):
    p = fraction_repaid
    q = 1 - p
    gini = 1 - (p ** 2 + q ** 2)
    if p == 0 or p == 1:
        entropy = 0.0
    else:
        entropy = -(p * np.log2(p) + q * np.log2(q))
    return gini, entropy

for fraction in [0.0, 0.2, 0.5, 0.8, 1.0]:
    gini, entropy = impurity_scores(fraction)
    print("fraction repaid =", fraction,
          "  gini =", round(gini, 3),
          "  entropy =", round(entropy, 3))

print("\nBoth are 0 for a pure group and biggest at the 50/50 mix.")

### Stretch (optional) — information gain for one split, by hand

Here is the tree's actual decision rule in numbers. Take the training applicants,
split them by one question ("is income above 50?"), and measure how much tidier
the two groups are than the whole. That drop in impurity is the **information
gain**. The tree tries every possible question and keeps the one with the biggest
gain.

In [ ]:
def group_entropy(labels):
    if len(labels) == 0:
        return 0.0
    fraction_repaid = labels.mean()
    _, entropy = impurity_scores(fraction_repaid)
    return entropy

income_train = X_train["income"]
before = group_entropy(y_train)

high_income = income_train > 50
labels_high = y_train[high_income]
labels_low = y_train[~high_income]

total = len(y_train)
after = (len(labels_high) / total) * group_entropy(labels_high) \
      + (len(labels_low) / total) * group_entropy(labels_low)

information_gain = before - after

print("entropy before the split :", round(before, 3))
print("entropy after  the split :", round(after, 3))
print("information gain          :", round(information_gain, 3))
print("\nA bigger gain means a better question.")

## What you just did

You grew a decision tree, read it as a flowchart, and saw exactly what impurity
and information gain mean. Most importantly, you saw that a deeper tree is not a
better tree: past a point it just memorises the training applicants and does worse
on new ones. Limiting the depth, called pruning, is one cure.

The next notebook shows a stronger cure: instead of fussing over one tree, use
many. Open `02_many_trees_beat_one.ipynb`.